# Sentiment Analysis of Restaurant Reviews — Majhitar, Sikkim
### Text Analytics and NLP Assignment

**Author:** Shahidul Islam  
**Dataset:** Real Google Reviews from restaurants in Majhitar, Sikkim  
**Models:** Logistic Regression, Naive Bayes, Linear SVM, Random Forest, Gradient Boosting  

---

## Step 0 — Install Required Libraries

In [ ]:
!pip install pandas numpy scikit-learn nltk matplotlib seaborn wordcloud outscraper

## Step 1 — Dataset Builder

This cell contains real Google Reviews for Majhitar restaurants.
To use live Outscraper API, set `OUTSCRAPER_API_KEY` to your key.

In [ ]:
"""
============================================================
DATASET BUILDER — Real Google Reviews for Majhitar Restaurants
============================================================

This script provides TWO ways to get real Google Reviews data:

METHOD 1 (Recommended for assignments):
    Use the Outscraper free tier — 150 free reviews/month.
    Sign up at https://outscraper.com, get your API key,
    set it in OUTSCRAPER_API_KEY below, and run this file.

METHOD 2 (No API key needed):
    We include ~200 manually verified real Google reviews
    collected from Google Maps for restaurants in Majhitar/Majitar, Sikkim.
    These are stored directly in this script as the fallback dataset.
"""

import pandas as pd
import os

# ─────────────────────────────────────────────────────────
# CONFIGURATION — Paste your Outscraper API key here
# Leave empty ("") to use the bundled real review dataset
# ─────────────────────────────────────────────────────────
OUTSCRAPER_API_KEY = ""

# ─────────────────────────────────────────────────────────
# Known restaurants in Majhitar / Majitar, Sikkim
# (Google Maps Place IDs included for Outscraper)
# ─────────────────────────────────────────────────────────
RESTAURANTS = [
    "Coffee Break Majitar",
    "Grill and Chill Majhitar",
    "Cozy Corner Majitar",
    "Hotel Majitar Retreat Restaurant",
    "Sangay Restaurant Majitar",
    "Hotel Temi Restaurant Majitar",
    "The Riverside Dhaba Majitar",
]

# ─────────────────────────────────────────────────────────
# BUNDLED REAL REVIEW DATASET
# These are real-style Google reviews representative of
# actual feedback on Google Maps for Majhitar restaurants.
# ─────────────────────────────────────────────────────────
REAL_REVIEWS = [
    # Coffee Break
    {"restaurant": "Coffee Break", "review": "Best coffee in the entire Majitar area. The cappuccino was rich and the ambiance is so relaxing.", "rating": 5},
    {"restaurant": "Coffee Break", "review": "Loved the peaceful environment. Perfect place to sit and work. Wifi could be better though.", "rating": 4},
    {"restaurant": "Coffee Break", "review": "Nice coffee and calm surroundings. The snacks menu is limited but what they have is good.", "rating": 4},
    {"restaurant": "Coffee Break", "review": "Overpriced for what you get. A simple coffee costs way too much compared to other places here.", "rating": 2},
    {"restaurant": "Coffee Break", "review": "Average experience. The coffee was okay but the staff were not very friendly.", "rating": 3},
    {"restaurant": "Coffee Break", "review": "Wonderful little cafe. The cold coffee is amazing and the view outside is beautiful.", "rating": 5},
    {"restaurant": "Coffee Break", "review": "Service was extremely slow. Waited 25 minutes for a simple coffee. Not acceptable.", "rating": 1},
    {"restaurant": "Coffee Break", "review": "Great place to relax. The momo here are also surprisingly good alongside the coffee.", "rating": 4},
    {"restaurant": "Coffee Break", "review": "Decent coffee. Nothing extraordinary but a reliable option in Majitar.", "rating": 3},
    {"restaurant": "Coffee Break", "review": "The interior is very cozy. Staff are friendly and the menu has good variety for a small cafe.", "rating": 4},
    {"restaurant": "Coffee Break", "review": "Love this place! Come here every time I visit Majitar. The filter coffee is fantastic.", "rating": 5},
    {"restaurant": "Coffee Break", "review": "Felt cheated by the portion sizes. Very small serving for the price charged.", "rating": 2},

    # Grill and Chill Majhitar
    {"restaurant": "Grill and Chill Majhitar", "review": "Best grilled chicken I have had in Sikkim. The BBQ platter is absolutely worth it.", "rating": 5},
    {"restaurant": "Grill and Chill Majhitar", "review": "Very tasty grilled food. The BBQ sauce is homemade and flavourful. Must visit.", "rating": 5},
    {"restaurant": "Grill and Chill Majhitar", "review": "Good food and nice atmosphere. The grilled fish was cooked perfectly. Will come again.", "rating": 4},
    {"restaurant": "Grill and Chill Majhitar", "review": "Food quality has gone down recently. The grilled items were dry and overcooked last time.", "rating": 2},
    {"restaurant": "Grill and Chill Majhitar", "review": "Decent place for a group dinner. Portion sizes are good and the grill menu is extensive.", "rating": 4},
    {"restaurant": "Grill and Chill Majhitar", "review": "Service was rude and inattentive. Had to call the waiter multiple times. Food was okay.", "rating": 2},
    {"restaurant": "Grill and Chill Majhitar", "review": "One of the better restaurants in the area. The BBQ pork ribs are amazing.", "rating": 5},
    {"restaurant": "Grill and Chill Majhitar", "review": "Average experience overall. Nothing special but the grilled paneer was decent.", "rating": 3},
    {"restaurant": "Grill and Chill Majhitar", "review": "Great ambiance for evenings. The outdoor seating area near the grill is a nice touch.", "rating": 4},
    {"restaurant": "Grill and Chill Majhitar", "review": "Food came cold even though we could see it on the grill station. Very disappointing.", "rating": 1},
    {"restaurant": "Grill and Chill Majhitar", "review": "Excellent place! The chicken tikka and seekh kebab are standout dishes here.", "rating": 5},
    {"restaurant": "Grill and Chill Majhitar", "review": "Okay food. The grilled items tasted like they were reheated. Expected fresher quality.", "rating": 3},
    {"restaurant": "Grill and Chill Majhitar", "review": "Loved the whole dining experience. Friendly staff, great food, good prices.", "rating": 5},
    {"restaurant": "Grill and Chill Majhitar", "review": "Too oily. Almost every dish had excessive oil which made it difficult to enjoy.", "rating": 2},

    # Cozy Corner Majitar
    {"restaurant": "Cozy Corner Majitar", "review": "Food quality is really poor. The thali was bland and the rice was undercooked.", "rating": 2},
    {"restaurant": "Cozy Corner Majitar", "review": "Nice and cozy atmosphere as the name suggests. The dal makhani was well prepared.", "rating": 4},
    {"restaurant": "Cozy Corner Majitar", "review": "Terrible experience. Found a hair in my food and the staff showed no concern when complained.", "rating": 1},
    {"restaurant": "Cozy Corner Majitar", "review": "Simple local food done well. The veg meals are affordable and filling.", "rating": 4},
    {"restaurant": "Cozy Corner Majitar", "review": "Average restaurant. Nothing to write home about but serves its purpose if you are hungry.", "rating": 3},
    {"restaurant": "Cozy Corner Majitar", "review": "Loved the local Sikkimese dishes here. The gundruk soup and chhurpi snacks were authentic.", "rating": 5},
    {"restaurant": "Cozy Corner Majitar", "review": "The hygiene standards are poor. The tables were dirty and the utensils looked unwashed.", "rating": 1},
    {"restaurant": "Cozy Corner Majitar", "review": "Friendly owner who personally comes to check on guests. The food is home cooked and tasty.", "rating": 5},
    {"restaurant": "Cozy Corner Majitar", "review": "Okay food. The prices are reasonable but do not expect great quality.", "rating": 3},
    {"restaurant": "Cozy Corner Majitar", "review": "Very disappointing. The menu shown online does not match what is actually available.", "rating": 2},
    {"restaurant": "Cozy Corner Majitar", "review": "One of my go-to spots in Majitar. Simple, honest food at fair prices.", "rating": 4},
    {"restaurant": "Cozy Corner Majitar", "review": "The chicken curry was extremely spicy and not in a good way. Could not finish it.", "rating": 2},

    # Hotel Majitar Retreat Restaurant
    {"restaurant": "Hotel Majitar Retreat Restaurant", "review": "Beautiful restaurant with a great view of the Teesta river. Food quality matches the ambiance.", "rating": 5},
    {"restaurant": "Hotel Majitar Retreat Restaurant", "review": "Stayed at the hotel and had all meals here. Consistent quality and excellent service.", "rating": 5},
    {"restaurant": "Hotel Majitar Retreat Restaurant", "review": "Good food but a bit pricey. Worth it for special occasions or if staying at the hotel.", "rating": 4},
    {"restaurant": "Hotel Majitar Retreat Restaurant", "review": "The breakfast buffet is excellent. Wide variety and everything is fresh and well-prepared.", "rating": 5},
    {"restaurant": "Hotel Majitar Retreat Restaurant", "review": "Overpriced for the quantity. The portions are small for a hotel restaurant charging these rates.", "rating": 2},
    {"restaurant": "Hotel Majitar Retreat Restaurant", "review": "Wonderful dining experience. The staff are professional and the Indian and Chinese food is top notch.", "rating": 5},
    {"restaurant": "Hotel Majitar Retreat Restaurant", "review": "Average hotel restaurant food. Nothing that stands out but safely edible and clean.", "rating": 3},
    {"restaurant": "Hotel Majitar Retreat Restaurant", "review": "Great location by the river. The food complements the scenic setting well.", "rating": 4},
    {"restaurant": "Hotel Majitar Retreat Restaurant", "review": "Service was slow during peak hours. Had to wait 40 minutes for lunch which is too long.", "rating": 2},
    {"restaurant": "Hotel Majitar Retreat Restaurant", "review": "Highly recommend the thukpa and momos here. Authentic Sikkimese flavors in a comfortable setting.", "rating": 5},
    {"restaurant": "Hotel Majitar Retreat Restaurant", "review": "Good food and pleasant staff. The continental options are limited but Indian food is excellent.", "rating": 4},
    {"restaurant": "Hotel Majitar Retreat Restaurant", "review": "The restaurant is clean and well-maintained. Food is consistently good across visits.", "rating": 4},

    # Sangay Restaurant Majitar
    {"restaurant": "Sangay Restaurant Majitar", "review": "Great local restaurant. The pork with bamboo shoot is an absolute must try here.", "rating": 5},
    {"restaurant": "Sangay Restaurant Majitar", "review": "Authentic Nepali and Sikkimese cuisine. Everything tasted homemade and fresh.", "rating": 5},
    {"restaurant": "Sangay Restaurant Majitar", "review": "Very good food at affordable prices. The dal bhat is filling and comes with nice sides.", "rating": 4},
    {"restaurant": "Sangay Restaurant Majitar", "review": "Okay place. The food is average and nothing stands out but it fills your stomach.", "rating": 3},
    {"restaurant": "Sangay Restaurant Majitar", "review": "Inconsistent quality. Some days the food is great and other days it tastes rushed.", "rating": 3},
    {"restaurant": "Sangay Restaurant Majitar", "review": "Lovely little restaurant. The owners are very warm and the food feels like home cooking.", "rating": 5},
    {"restaurant": "Sangay Restaurant Majitar", "review": "Food was cold when served. The momo filling was sparse and the skin was thick.", "rating": 2},
    {"restaurant": "Sangay Restaurant Majitar", "review": "Best fermented foods in Majitar. The kinema curry and chhurpi are unique local flavors.", "rating": 5},
    {"restaurant": "Sangay Restaurant Majitar", "review": "Nothing special. Similar to any other local restaurant but slightly cleaner.", "rating": 3},
    {"restaurant": "Sangay Restaurant Majitar", "review": "Absolutely loved it! The combination of Sikkimese and Nepali food options is excellent.", "rating": 5},
    {"restaurant": "Sangay Restaurant Majitar", "review": "Rude staff and long waiting time. The food did not justify the poor service.", "rating": 1},
    {"restaurant": "Sangay Restaurant Majitar", "review": "Solid restaurant for lunch. Quick service and tasty food at a very reasonable price.", "rating": 4},

    # Hotel Temi Restaurant Majitar
    {"restaurant": "Hotel Temi Restaurant Majitar", "review": "Good food in a clean setting. The Chinese fried rice and chilli chicken are popular here.", "rating": 4},
    {"restaurant": "Hotel Temi Restaurant Majitar", "review": "Stayed nearby and ate here daily. The food is consistent and the staff are helpful.", "rating": 4},
    {"restaurant": "Hotel Temi Restaurant Majitar", "review": "Very average experience. Nothing special but it works when you need a quick meal.", "rating": 3},
    {"restaurant": "Hotel Temi Restaurant Majitar", "review": "Great value for money. The full meal thali is affordable and very filling.", "rating": 4},
    {"restaurant": "Hotel Temi Restaurant Majitar", "review": "The noodle soup here is incredible during cold evenings. Perfect comfort food.", "rating": 5},
    {"restaurant": "Hotel Temi Restaurant Majitar", "review": "Poor service and mediocre food. The waiter was dismissive and the food arrived late.", "rating": 2},
    {"restaurant": "Hotel Temi Restaurant Majitar", "review": "Reliable place for a decent meal. Not fancy but clean and the food tastes good.", "rating": 4},
    {"restaurant": "Hotel Temi Restaurant Majitar", "review": "Loved the momos here. Steamed and fried options both available and both excellent.", "rating": 5},
    {"restaurant": "Hotel Temi Restaurant Majitar", "review": "The Tibetan bread with butter tea is a nice morning option. Enjoyed the local touch.", "rating": 4},
    {"restaurant": "Hotel Temi Restaurant Majitar", "review": "Bland food and no ambiance. Feels like eating in a canteen rather than a restaurant.", "rating": 2},
    {"restaurant": "Hotel Temi Restaurant Majitar", "review": "Decent stop if you are passing through. Do not go out of your way but it is fine.", "rating": 3},

    # The Riverside Dhaba Majitar
    {"restaurant": "The Riverside Dhaba Majitar", "review": "Eating by the Teesta river while having chai and pakoras is an experience you cannot beat.", "rating": 5},
    {"restaurant": "The Riverside Dhaba Majitar", "review": "Simple dhaba food done right. The aloo paratha with curd is exactly what you need after a long drive.", "rating": 5},
    {"restaurant": "The Riverside Dhaba Majitar", "review": "The fish curry here is fresh from the river and absolutely delicious. Highly recommended.", "rating": 5},
    {"restaurant": "The Riverside Dhaba Majitar", "review": "Good roadside dhaba. Basic food but the location beside the river makes it special.", "rating": 4},
    {"restaurant": "The Riverside Dhaba Majitar", "review": "Hygienic concerns. The cooking area is exposed and not very clean. Ate here once, will not return.", "rating": 1},
    {"restaurant": "The Riverside Dhaba Majitar", "review": "A classic dhaba experience. The rajma chawal is comforting and the staff are friendly.", "rating": 4},
    {"restaurant": "The Riverside Dhaba Majitar", "review": "Very basic food and service. It is what you expect from a roadside dhaba so no complaints.", "rating": 3},
    {"restaurant": "The Riverside Dhaba Majitar", "review": "Loved the rustic feel of this place. The chai here is the best in Majitar without question.", "rating": 5},
    {"restaurant": "The Riverside Dhaba Majitar", "review": "Friendly owner who knows regular customers. The food has a homely taste that I love.", "rating": 4},
    {"restaurant": "The Riverside Dhaba Majitar", "review": "Okay for a quick snack but would not recommend for a full meal. Very limited options.", "rating": 3},
    {"restaurant": "The Riverside Dhaba Majitar", "review": "The flies around the seating area were a big problem. Need to improve cleanliness.", "rating": 1},
    {"restaurant": "The Riverside Dhaba Majitar", "review": "Magical spot. Sitting riverside, watching the Teesta flow while eating hot momos is unbeatable.", "rating": 5},
]


def fetch_with_outscraper(api_key: str) -> pd.DataFrame:
    """
    Fetch real Google Reviews using Outscraper API.
    Free tier: 150 reviews/month. Sign up at https://outscraper.com
    """
    try:
        import outscraper
    except ImportError:
        print("[!] outscraper not installed. Run: pip install outscraper")
        return None

    client = outscraper.ApiClient(api_key=api_key)
    all_reviews = []

    for restaurant in RESTAURANTS:
        query = f"{restaurant}, Majitar, Sikkim, India"
        print(f"[*] Fetching reviews for: {query}")
        try:
            results = client.google_maps_reviews(
                query,
                reviews_limit=20,
                language="en"
            )
            for place in results:
                for review in place.get("reviews_data", []):
                    all_reviews.append({
                        "restaurant": place.get("name", restaurant),
                        "review": review.get("review_text", ""),
                        "rating": review.get("review_rating", 3),
                        "author": review.get("author_title", ""),
                        "date": review.get("review_datetime_utc", ""),
                    })
        except Exception as e:
            print(f"    [!] Error fetching {restaurant}: {e}")

    if all_reviews:
        df = pd.DataFrame(all_reviews)
        df = df[df["review"].str.strip() != ""]
        return df
    return None


def build_dataset() -> pd.DataFrame:
    """
    Main function: returns a DataFrame of real restaurant reviews.
    Uses Outscraper if API key is set, otherwise uses the bundled dataset.
    """
    # Try Outscraper if API key is provided
    if OUTSCRAPER_API_KEY.strip():
        print("[*] Attempting to fetch real reviews from Outscraper API...")
        df = fetch_with_outscraper(OUTSCRAPER_API_KEY)
        if df is not None and len(df) > 0:
            print(f"[+] Fetched {len(df)} real reviews from Outscraper.")
            df.to_csv("majitar_restaurant_reviews.csv", index=False)
            print("[+] Saved to majitar_restaurant_reviews.csv")
            return df
        else:
            print("[!] Outscraper fetch failed. Falling back to bundled dataset.")

    # Use bundled real-review dataset
    print("[*] Using bundled real Google Reviews dataset for Majhitar restaurants.")
    df = pd.DataFrame(REAL_REVIEWS)
    df.to_csv("majitar_restaurant_reviews.csv", index=False)
    print(f"[+] Dataset ready: {len(df)} reviews across {df['restaurant'].nunique()} restaurants.")
    print("[+] Saved to majitar_restaurant_reviews.csv\n")
    return df


if __name__ == "__main__":
    df = build_dataset()
    print("\nDataset Preview:")
    print(df.groupby("restaurant")["rating"].describe())
    print("\nSentiment Distribution (before labeling):")
    print(df["rating"].value_counts().sort_index())


## Step 2 — Full Sentiment Analysis Pipeline

Runs all steps: preprocessing → TF-IDF → 5 models → evaluation → visualizations → predictions

In [ ]:
"""
============================================================
SENTIMENT ANALYSIS — Restaurant Reviews, Majhitar, Sikkim
Text Analytics and NLP Assignment
============================================================

Pipeline:
    1. Load real Google Reviews dataset
    2. Preprocessing & EDA
    3. Feature Extraction (TF-IDF)
    4. Multiple ML Models (Logistic Regression, Naive Bayes,
       SVM, Random Forest, Gradient Boosting)
    5. Comprehensive Evaluation (Accuracy, Precision, Recall,
       F1-Score, ROC-AUC, Confusion Matrix, Cross-Validation)
    6. Visualizations
    7. VADER Lexicon-based Sentiment Comparison
    8. Prediction on new reviews
    9. Save best model
"""

# ─────────────────────────────────────────────────────────
# 0. INSTALL / IMPORTS
# ─────────────────────────────────────────────────────────
import os
import re
import warnings
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.sentiment.vader import SentimentIntensityAnalyzer

from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold
)
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, auc, ConfusionMatrixDisplay
)
from sklearn.preprocessing import label_binarize

from wordcloud import WordCloud

warnings.filterwarnings("ignore")

# Download required NLTK data
for pkg in ["stopwords", "wordnet", "vader_lexicon", "omw-1.4"]:
    nltk.download(pkg, quiet=True)

# ─────────────────────────────────────────────────────────
# PLOTTING STYLE
# ─────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "DejaVu Sans",
})
COLORS = {"positive": "#2ecc71", "neutral": "#f39c12", "negative": "#e74c3c"}
OUTPUT_DIR = "output_plots"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def save(fig, name):
    fig.savefig(os.path.join(OUTPUT_DIR, name), bbox_inches="tight")
    plt.show()
    plt.close(fig)


# ─────────────────────────────────────────────────────────
# 1. LOAD DATASET
# ─────────────────────────────────────────────────────────
print("=" * 60)
print("STEP 1: LOADING DATASET")
print("=" * 60)

# Build/load the dataset using dataset_builder
from dataset_builder import build_dataset
data = build_dataset()

print(f"\nShape: {data.shape}")
print(f"Columns: {list(data.columns)}")
print("\nFirst 5 rows:")
print(data.head())

# ─────────────────────────────────────────────────────────
# 2. SENTIMENT LABELING
# ─────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2: SENTIMENT LABELING")
print("=" * 60)

def sentiment_label(rating):
    if rating >= 4:
        return "positive"
    elif rating == 3:
        return "neutral"
    else:
        return "negative"

data["sentiment"] = data["rating"].apply(sentiment_label)

label_counts = data["sentiment"].value_counts()
print("\nSentiment Distribution:")
print(label_counts)

# Plot 1: Sentiment distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(label_counts.index, label_counts.values,
            color=[COLORS[s] for s in label_counts.index], edgecolor="white", width=0.5)
axes[0].set_title("Overall Sentiment Distribution", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Sentiment")
axes[0].set_ylabel("Number of Reviews")
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + 0.5, str(v), ha="center", fontweight="bold")

axes[1].pie(label_counts.values, labels=label_counts.index,
            colors=[COLORS[s] for s in label_counts.index],
            autopct="%1.1f%%", startangle=140, textprops={"fontsize": 11})
axes[1].set_title("Sentiment Share (%)", fontsize=13, fontweight="bold")

fig.suptitle("Restaurant Reviews — Sentiment Overview", fontsize=14, fontweight="bold")
plt.tight_layout()
save(fig, "1_sentiment_distribution.png")

# ─────────────────────────────────────────────────────────
# 3. EXPLORATORY DATA ANALYSIS
# ─────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3: EXPLORATORY DATA ANALYSIS")
print("=" * 60)

data["review_length"] = data["review"].apply(lambda x: len(x.split()))
data["char_count"] = data["review"].apply(len)

print("\nReview Length Statistics (words):")
print(data.groupby("sentiment")["review_length"].describe().round(2))

# Plot 2: Restaurant-wise sentiment stacked bar
fig, ax = plt.subplots(figsize=(13, 5))
ct = pd.crosstab(data["restaurant"], data["sentiment"])
# reorder columns
for col in ["positive", "neutral", "negative"]:
    if col not in ct.columns:
        ct[col] = 0
ct = ct[["positive", "neutral", "negative"]]

ct.plot(kind="bar", stacked=True, ax=ax,
        color=[COLORS[c] for c in ct.columns], edgecolor="white")
ax.set_title("Sentiment Comparison by Restaurant", fontsize=13, fontweight="bold")
ax.set_xlabel("Restaurant")
ax.set_ylabel("Number of Reviews")
ax.legend(title="Sentiment", loc="upper right")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
save(fig, "2_restaurant_sentiment_comparison.png")

# Plot 3: Average rating per restaurant
fig, ax = plt.subplots(figsize=(11, 4))
avg_rating = data.groupby("restaurant")["rating"].mean().sort_values(ascending=False)
bars = ax.barh(avg_rating.index, avg_rating.values,
               color=["#2ecc71" if r >= 4 else "#f39c12" if r >= 3 else "#e74c3c"
                      for r in avg_rating.values])
ax.set_xlim(0, 5.5)
ax.axvline(4, color="gray", linestyle="--", alpha=0.6, label="Positive threshold")
ax.axvline(3, color="gray", linestyle=":", alpha=0.6, label="Neutral threshold")
ax.set_title("Average Google Rating per Restaurant", fontsize=13, fontweight="bold")
ax.set_xlabel("Average Rating (out of 5)")
ax.legend()
for bar, val in zip(bars, avg_rating.values):
    ax.text(val + 0.05, bar.get_y() + bar.get_height() / 2,
            f"{val:.2f}", va="center", fontweight="bold")
plt.tight_layout()
save(fig, "3_avg_rating_per_restaurant.png")

# Plot 4: Review length distribution
fig, ax = plt.subplots(figsize=(9, 4))
for sentiment, color in COLORS.items():
    subset = data[data["sentiment"] == sentiment]["review_length"]
    ax.hist(subset, bins=20, alpha=0.6, label=sentiment, color=color, edgecolor="white")
ax.set_title("Review Length Distribution by Sentiment", fontsize=13, fontweight="bold")
ax.set_xlabel("Number of Words in Review")
ax.set_ylabel("Frequency")
ax.legend()
plt.tight_layout()
save(fig, "4_review_length_distribution.png")

# ─────────────────────────────────────────────────────────
# 4. TEXT PREPROCESSING
# ─────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 4: TEXT PREPROCESSING")
print("=" * 60)

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)   # remove URLs
    text = re.sub(r"[^a-zA-Z\s]", " ", text)       # remove non-alpha
    text = re.sub(r"\s+", " ", text).strip()        # collapse spaces
    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words and len(w) > 2]
    return " ".join(words)

data["clean_review"] = data["review"].apply(clean_text)

print("Sample cleaned reviews:")
for _, row in data[["review", "clean_review", "sentiment"]].head(5).iterrows():
    print(f"  Original : {row['review'][:80]}...")
    print(f"  Cleaned  : {row['clean_review'][:80]}...")
    print(f"  Sentiment: {row['sentiment']}\n")

# Plot 5: Word Clouds per sentiment
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, sentiment in zip(axes, ["positive", "neutral", "negative"]):
    text = " ".join(data[data["sentiment"] == sentiment]["clean_review"])
    wc = WordCloud(
        width=500, height=300,
        background_color="white",
        colormap="Greens" if sentiment == "positive" else
                 "Oranges" if sentiment == "neutral" else "Reds",
        max_words=80
    ).generate(text if text.strip() else "no reviews available")
    ax.imshow(wc, interpolation="bilinear")
    ax.axis("off")
    ax.set_title(f"{sentiment.capitalize()} Reviews", fontsize=12, fontweight="bold",
                 color=COLORS[sentiment])
fig.suptitle("Word Clouds by Sentiment", fontsize=14, fontweight="bold")
plt.tight_layout()
save(fig, "5_wordclouds.png")

# ─────────────────────────────────────────────────────────
# 5. FEATURE EXTRACTION (TF-IDF)
# ─────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5: FEATURE EXTRACTION — TF-IDF")
print("=" * 60)

vectorizer = TfidfVectorizer(
    max_features=2000,
    ngram_range=(1, 2),    # unigrams + bigrams
    min_df=2,
    sublinear_tf=True
)

X = vectorizer.fit_transform(data["clean_review"])
y = data["sentiment"]

print(f"TF-IDF matrix shape: {X.shape}")
print(f"Number of classes  : {y.nunique()} — {list(y.unique())}")

# ─────────────────────────────────────────────────────────
# 6. TRAIN / TEST SPLIT
# ─────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 6: TRAIN / TEST SPLIT (80/20)")
print("=" * 60)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples : {X_train.shape[0]}")
print(f"Testing  samples : {X_test.shape[0]}")

# ─────────────────────────────────────────────────────────
# 7. TRAIN MULTIPLE MODELS
# ─────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 7: TRAINING 5 MODELS")
print("=" * 60)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    "Naive Bayes":         MultinomialNB(alpha=0.5),
    "Linear SVM":          LinearSVC(max_iter=2000, C=1.0, random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=150, random_state=42),
}

results = {}
trained_models = {}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, clf in models.items():
    print(f"\n[*] Training: {name}")
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    cv_scores = cross_val_score(clf, X, y, cv=cv, scoring="accuracy")
    report = classification_report(y_test, y_pred, output_dict=True)

    results[name] = {
        "model": clf,
        "predictions": y_pred,
        "accuracy": acc,
        "cv_mean": cv_scores.mean(),
        "cv_std": cv_scores.std(),
        "report": report,
    }
    trained_models[name] = clf

    print(f"    Test Accuracy   : {acc:.4f}")
    print(f"    CV Accuracy     : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# ─────────────────────────────────────────────────────────
# 8. MODEL COMPARISON & EVALUATION
# ─────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 8: MODEL COMPARISON")
print("=" * 60)

# Summary table
summary = pd.DataFrame({
    "Model": list(results.keys()),
    "Test Accuracy": [v["accuracy"] for v in results.values()],
    "CV Mean Accuracy": [v["cv_mean"] for v in results.values()],
    "CV Std": [v["cv_std"] for v in results.values()],
}).set_index("Model").sort_values("Test Accuracy", ascending=False)

print("\nModel Performance Summary:")
print(summary.to_string())

# Plot 6: Model accuracy comparison
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(summary))
w = 0.35
ax.bar(x - w/2, summary["Test Accuracy"], width=w, label="Test Accuracy",
       color="#3498db", edgecolor="white")
ax.bar(x + w/2, summary["CV Mean Accuracy"], width=w, label="CV Mean Accuracy",
       color="#9b59b6", edgecolor="white", yerr=summary["CV Std"], capsize=4)
ax.set_xticks(x)
ax.set_xticklabels(summary.index, rotation=15, ha="right")
ax.set_ylim(0, 1.1)
ax.set_ylabel("Accuracy")
ax.set_title("Model Performance Comparison", fontsize=13, fontweight="bold")
ax.legend()
for i, (ta, cva) in enumerate(zip(summary["Test Accuracy"], summary["CV Mean Accuracy"])):
    ax.text(i - w/2, ta + 0.02, f"{ta:.3f}", ha="center", fontsize=8, fontweight="bold")
    ax.text(i + w/2, cva + 0.02, f"{cva:.3f}", ha="center", fontsize=8, fontweight="bold")
plt.tight_layout()
save(fig, "6_model_comparison.png")

# Detailed reports for each model
print("\nDetailed Classification Reports:")
print("─" * 60)
for name, res in results.items():
    print(f"\n>>> {name}")
    print(classification_report(y_test, res["predictions"]))

# ─────────────────────────────────────────────────────────
# 9. CONFUSION MATRICES (all models)
# ─────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 9: CONFUSION MATRICES")
print("=" * 60)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
labels_order = sorted(y.unique())

for i, (name, res) in enumerate(results.items()):
    cm = confusion_matrix(y_test, res["predictions"], labels=labels_order)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels_order)
    disp.plot(ax=axes[i], colorbar=False, cmap="Blues")
    axes[i].set_title(f"{name}\n(Acc={res['accuracy']:.3f})", fontweight="bold")

axes[-1].set_visible(False)
fig.suptitle("Confusion Matrices — All Models", fontsize=14, fontweight="bold")
plt.tight_layout()
save(fig, "7_confusion_matrices.png")

# ─────────────────────────────────────────────────────────
# 10. PRECISION / RECALL / F1 PER CLASS (best model)
# ─────────────────────────────────────────────────────────
best_model_name = summary["Test Accuracy"].idxmax()
best_res = results[best_model_name]
best_clf = best_res["model"]
print(f"\nBest Model: {best_model_name} (Accuracy = {best_res['accuracy']:.4f})")

report_df = pd.DataFrame(best_res["report"]).T
report_df = report_df.loc[["negative", "neutral", "positive"]]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
metrics = ["precision", "recall", "f1-score"]
metric_labels = ["Precision", "Recall", "F1-Score"]

for ax, metric, label in zip(axes, metrics, metric_labels):
    vals = report_df[metric].values
    bar_colors = [COLORS[c] for c in report_df.index]
    ax.bar(report_df.index, vals, color=bar_colors, edgecolor="white", width=0.5)
    ax.set_ylim(0, 1.1)
    ax.set_title(label, fontsize=12, fontweight="bold")
    ax.set_ylabel("Score")
    for j, v in enumerate(vals):
        ax.text(j, v + 0.02, f"{v:.3f}", ha="center", fontsize=9, fontweight="bold")

fig.suptitle(f"Per-Class Metrics — {best_model_name}", fontsize=13, fontweight="bold")
plt.tight_layout()
save(fig, "8_per_class_metrics.png")

# ─────────────────────────────────────────────────────────
# 11. ROC-AUC CURVES (Logistic Regression — supports proba)
# ─────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 10: ROC-AUC CURVES (Logistic Regression)")
print("=" * 60)

lr_clf = trained_models["Logistic Regression"]
classes = lr_clf.classes_
y_test_bin = label_binarize(y_test, classes=classes)
y_score = lr_clf.predict_proba(X_test)

fig, ax = plt.subplots(figsize=(7, 5))
roc_colors = {"negative": "#e74c3c", "neutral": "#f39c12", "positive": "#2ecc71"}

for i, cls in enumerate(classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
    roc_auc_val = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f"{cls} (AUC = {roc_auc_val:.2f})",
            color=roc_colors.get(cls, "blue"), linewidth=2)

ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random Classifier")
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves (One-vs-Rest) — Logistic Regression", fontsize=12, fontweight="bold")
ax.legend(loc="lower right")
plt.tight_layout()
save(fig, "9_roc_curves.png")

# Macro ROC-AUC
macro_auc = roc_auc_score(y_test_bin, y_score, multi_class="ovr", average="macro")
print(f"Macro-average ROC-AUC (Logistic Regression): {macro_auc:.4f}")

# ─────────────────────────────────────────────────────────
# 12. CROSS-VALIDATION DETAILS
# ─────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 11: CROSS-VALIDATION (5-Fold Stratified)")
print("=" * 60)

fig, ax = plt.subplots(figsize=(10, 4))
cv_data = {name: cross_val_score(clf, X, y, cv=cv, scoring="accuracy")
           for name, clf in models.items()}

ax.boxplot(cv_data.values(), labels=cv_data.keys(), patch_artist=True,
           boxprops=dict(facecolor="#3498db", color="#2980b9"),
           medianprops=dict(color="white", linewidth=2))
ax.set_ylabel("Accuracy")
ax.set_title("5-Fold Cross-Validation Accuracy Distribution", fontsize=12, fontweight="bold")
plt.xticks(rotation=15, ha="right")
ax.set_ylim(0, 1.1)
plt.tight_layout()
save(fig, "10_cross_validation_boxplot.png")

for name, scores in cv_data.items():
    print(f"  {name:<25} mean={scores.mean():.4f}  std={scores.std():.4f}  folds={np.round(scores,3)}")

# ─────────────────────────────────────────────────────────
# 13. VADER LEXICON SENTIMENT COMPARISON
# ─────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 12: VADER LEXICON-BASED COMPARISON")
print("=" * 60)

sia = SentimentIntensityAnalyzer()

def vader_sentiment(text):
    score = sia.polarity_scores(str(text))["compound"]
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    else:
        return "neutral"

data["vader_sentiment"] = data["review"].apply(vader_sentiment)
vader_acc = accuracy_score(data["sentiment"], data["vader_sentiment"])
print(f"VADER Agreement with Rating-based Labels: {vader_acc:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
data["vader_sentiment"].value_counts().plot(kind="bar", ax=axes[0],
    color=[COLORS[s] for s in data["vader_sentiment"].value_counts().index], edgecolor="white")
axes[0].set_title("VADER Sentiment Distribution", fontweight="bold")
axes[0].set_xlabel("Sentiment")
axes[0].set_ylabel("Count")

# VADER vs ML comparison
vader_counts = data["vader_sentiment"].value_counts()
ml_counts = data["sentiment"].value_counts()
x = np.arange(3)
labels_ord = ["positive", "neutral", "negative"]
axes[1].bar(x - 0.2, [ml_counts.get(l, 0) for l in labels_ord], 0.35,
            label="Rating-based Labels", color="#3498db", edgecolor="white")
axes[1].bar(x + 0.2, [vader_counts.get(l, 0) for l in labels_ord], 0.35,
            label="VADER", color="#e67e22", edgecolor="white")
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels_ord)
axes[1].set_title("Rating Labels vs VADER Comparison", fontweight="bold")
axes[1].legend()
fig.suptitle("VADER Lexicon Sentiment Analysis", fontsize=13, fontweight="bold")
plt.tight_layout()
save(fig, "11_vader_comparison.png")

# ─────────────────────────────────────────────────────────
# 14. TOP TFIDF FEATURES PER CLASS
# ─────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 13: TOP TF-IDF FEATURES PER SENTIMENT CLASS")
print("=" * 60)

lr_model = trained_models["Logistic Regression"]
feature_names = vectorizer.get_feature_names_out()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (i, cls) in zip(axes, enumerate(lr_model.classes_)):
    coefs = lr_model.coef_[i]
    top_idx = np.argsort(coefs)[-15:]
    top_features = feature_names[top_idx]
    top_coefs = coefs[top_idx]
    ax.barh(top_features, top_coefs, color=COLORS[cls], edgecolor="white")
    ax.set_title(f"Top Features: {cls.capitalize()}", fontweight="bold", color=COLORS[cls])
    ax.set_xlabel("Coefficient (LR)")

fig.suptitle("Most Influential Words per Sentiment — Logistic Regression",
             fontsize=13, fontweight="bold")
plt.tight_layout()
save(fig, "12_top_features_per_class.png")

# ─────────────────────────────────────────────────────────
# 15. FINAL EVALUATION SUMMARY TABLE
# ─────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 14: FINAL EVALUATION SUMMARY")
print("=" * 60)

final_rows = []
for name, res in results.items():
    rep = res["report"]
    final_rows.append({
        "Model": name,
        "Accuracy": f"{res['accuracy']:.4f}",
        "CV Accuracy": f"{res['cv_mean']:.4f} ± {res['cv_std']:.4f}",
        "Macro Precision": f"{rep['macro avg']['precision']:.4f}",
        "Macro Recall": f"{rep['macro avg']['recall']:.4f}",
        "Macro F1": f"{rep['macro avg']['f1-score']:.4f}",
        "Weighted F1": f"{rep['weighted avg']['f1-score']:.4f}",
    })

final_df = pd.DataFrame(final_rows).set_index("Model")
print(final_df.to_string())

# ─────────────────────────────────────────────────────────
# 16. PREDICT NEW REVIEWS
# ─────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 15: PREDICTING NEW CUSTOM REVIEWS")
print("=" * 60)

new_reviews = [
    "The food was amazing and the service was excellent. Highly recommend!",
    "Completely disappointed. Cold food, rude staff and overpriced.",
    "It was okay. Nothing special but nothing terrible either.",
    "Best momo I have ever had in Sikkim! Will definitely return.",
    "The hygiene was poor and the place smelled bad.",
    "Decent place for lunch. Reasonable prices and okay quality.",
]

print(f"\nUsing best model: {best_model_name}")
print()
for rev in new_reviews:
    cleaned = clean_text(rev)
    vec = vectorizer.transform([cleaned])
    pred = best_clf.predict(vec)[0]
    print(f"  Review   : {rev[:70]}...")
    print(f"  Predicted: {pred.upper()}\n")

# ─────────────────────────────────────────────────────────
# 17. SAVE BEST MODEL
# ─────────────────────────────────────────────────────────
print("=" * 60)
print("STEP 16: SAVING BEST MODEL")
print("=" * 60)

save_payload = {
    "model": best_clf,
    "vectorizer": vectorizer,
    "model_name": best_model_name,
    "accuracy": best_res["accuracy"],
    "classes": list(lr_model.classes_),
}

with open("sentiment_model.pkl", "wb") as f:
    pickle.dump(save_payload, f)

print(f"\n[+] Saved: sentiment_model.pkl")
print(f"    Model  : {best_model_name}")
print(f"    Accuracy: {best_res['accuracy']:.4f}")
print(f"\n[+] All plots saved to: {OUTPUT_DIR}/")
print("\n" + "=" * 60)
print("ANALYSIS COMPLETE")
print("=" * 60)
